In [ ]:
import ast
import os
from bs4 import BeautifulSoup
from pypdf import PdfReader
from ollama import chat
import re
from pywebio.input import input, TEXT
from pywebio.output import put_text, use_scope, put_table, put_markdown, put_file, put_loading
from pathlib import Path

# ==========================================================
# 1️⃣ Load PDFs as full text
# ==========================================================

#pdf_files = {
#   "2025_regs": "sporting_regulations_f3_2025.pdf",
#    "2026_regs": "sporting_regulations_f3_2026.pdf",
#    "driving_standards": "DrivingStandards.pdf",
#    "sporting_code": "SportingCode.pdf",
#    "penalty_guidelines": "PenaltyGuideLines.pdf"
#}
#
#documents = {}

#print("Loading PDFs...")

#for year, filename in pdf_files.items():
#    print(f"Processing {filename}...")
#    reader = PdfReader(filename)
#    text = ""

#    for page in reader.pages:
#        page_text = page.extract_text()
#        if page_text:
#            text += page_text + "\n"

#    documents[year] = text
#    print(f" PDF loaded for {year}, length: {len(text)} characters")

#pattern = r'^(?:ARTICLE\s+\d+.*|\d+(?:\.\d+)+.*)'
#headings_2025 = re.findall(pattern, documents["2025_regs"], re.MULTILINE)
#headings_2026 = re.findall(pattern, documents["2026_regs"], re.MULTILINE)
#headings_driving_standards= re.findall(pattern, documents["driving_standards"], re.MULTILINE)
#headings_sporting_code= re.findall(pattern, documents["sporting_code"], re.MULTILINE)
#headings_penalty_guidelines= re.findall(pattern, documents["penalty_guidelines"], re.MULTILINE)

# ==========================================================
# 2️⃣ Load question from email
# ==========================================================

import sys

# ==========================================================
# 2️⃣ Load question (HTML OR terminal)
# ==========================================================

def get_user_question(prompt_text="Type your question:"):
    """
    Prompt the user for a question directly.
    Returns the entered question as a string.
    """
    question = "Compare all sections"
    if not question:
        raise ValueError("No question provided.")
    return question


# Example usage
user_question = get_user_question("Enter the question about F3 regulations: ")
print(f"\nQuestion: {user_question[:100]}...\n")

# ==========================================================
# 3️⃣ Reason about relevant sections based on question
# ==========================================================


# ==========================================================
# 4️⃣ Load GGUF 8-bit model
# ==========================================================

GGUF_MODEL_PATH = r"C:\Users\ArjanFaberVAR\OneDrive - Van Amersfoort Racing B.V\Regulations-VAR - Documenten\Meta-Llama-3-8B-Instruct.Q4_0.gguf"
n_threads = os.cpu_count() - 1

import subprocess
import time

# Start Ollama server
OLLAMA_EXE = r"C:\Users\ArjanFaberVAR\AppData\Local\Programs\Ollama\Ollama.exe"
MODEL_NAME = "gpt-oss:120b-cloud"
API_KEY = "c109ad8c287f14ba1be1c61a0e09fc1fe.tU6J3r7fqva9UKVFYjPRghw2"

os.environ["OLLAMA_API_KEY"] = API_KEY


ModuleNotFoundError: No module named 'pypdf'

In [ ]:

import builtins
# ==========================================================
# Ask question from terminal
# ==========================================================

def get_user_question(prompt_text="Enter your question: "):
    question = builtins.input(prompt_text).strip()

    if not question:
        raise ValueError("No question provided.")

    return question


# ==========================================================
# Ensure model exists
# ==========================================================

def ensure_model_exists():
    try:
        result = subprocess.run(
            [OLLAMA_EXE, "list"],
            capture_output=True,
            text=True
        )

        if MODEL_NAME not in result.stdout:
            print(f"Model '{MODEL_NAME}' not found.")
            print("Downloading model...")

            subprocess.run(
                [OLLAMA_EXE, "pull", MODEL_NAME],
                check=True
            )

            print("Model downloaded successfully.")
        else:
            print(f"Model '{MODEL_NAME}' already exists.")

    except Exception as e:
        print("Error while checking model:")
        print(e)
        raise


# ==========================================================
# Start Ollama server
# ==========================================================

def start_ollama():
    try:
        subprocess.Popen(
            [OLLAMA_EXE, "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        print("Starting Ollama server...")
        time.sleep(5)
        print("Ollama server ready.")

    except Exception as e:
        print("Failed to start Ollama server:")
        print(e)
        raise


# ==========================================================
# Ask Ollama
# ==========================================================

def ask_ollama(prompt, system=None,
               max_tokens=800,
               temperature=0.0):

    messages = []

    if system:
        messages.append({
            "role": "system",
            "content": system
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    response = chat(
        model=MODEL_NAME,
        messages=messages,
        options={
            "temperature": temperature,
            "num_predict": max_tokens
        }
    )

    return response["message"]["content"]


# ==========================================================
# Main
# ==========================================================

def main():

    ensure_model_exists()
    start_ollama()

    user_question = get_user_question(
        "\nEnter your question about F3 regulations:\n> "
    )

    print("\nQuestion:")
    print(user_question)

    reason_prompt = f"""
You are an expert on Formula 3 regulations.

Answer the following question clearly and precisely:

{user_question}
"""

    print("\nSending request to model...\n")

    answer = ask_ollama(
        reason_prompt,
        max_tokens=1000,
        temperature=0.0
    )

    print("\n================ ANSWER ================\n")
    print(answer)
    print("\n========================================\n")


if __name__ == "__main__":
    main()
```
